# Is the per-perturbation neural bridge training?

Out-of-batch prediction on Replogle K562 with `NeuralPerXEmbeddingBridge` (a separate $A_x$ per perturbation, only $\psi$ shared). Trains on all batches except `N_HELD_OUT`, and records at **every step**:

- **training loss**: the cross-fitted stage-2 loss that AdamW actually minimises (over random folds of the training batches, so it is noisy);
- **held-out loss**: the same cell-weighted squared error on the held-out batches, with $A_x$ solved on all training batches and the proxies being the held-out batches' seen controls.

Both use unscaled expression targets and the same $\psi$ at each step. Only control inputs are standardised, using training-control statistics. Step 0 is the untrained $\psi$, so the held-out curve shows what training adds over random features.

Needs the Replogle file and a GPU with about 96 GB of RAM (as `hpc/run.sbatch`).

In [ ]:
import sys
from functools import cache
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from genetic_perturbation_playground.data.data_loader import DATA_DIR, load_dataset
from genetic_perturbation_playground.model.batch_transfer import (
    NeuralBridgeTransfer,
    Standardised,
    _bridge_data,
    _positions,
    batch_labels,
    is_control,
)
from genetic_perturbation_playground.model.proxy_unconfounded_neural import (
    NeuralPerXEmbeddingBridge,
)

sys.path.insert(0, str(DATA_DIR.parent / "scripts"))
from train_batch_transfer_expression import _fit_linear, _seen_controls, _split_controls, score

N_HELD_OUT = 5
SEED = 0
TOP_K = 20
NEURAL = {  # the settings tune_replogle_perx selected most often; n_steps its largest checkpoint
    "reg": 1e-5,
    "weight_decay": 0.0,
    "n_features_w": 8,
    "width_w": 8,
    "depth": 2,
    "lr": 1e-3,
    "n_steps": 400,
    "cross_fit_folds": 5,
    "seed": SEED,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

## Split

As in `cross_validate`: every batch's controls are split in two halves, and the models only ever see the `seen` half.

In [ ]:
adata = load_dataset()
batch = batch_labels(adata)
rng = np.random.default_rng(SEED)
held_out = rng.choice(np.unique(batch), N_HELD_OUT, replace=False)
seen = _seen_controls(adata, batch, rng)

is_held_out = np.isin(batch, held_out)
train = adata[~is_held_out & (~is_control(adata) | seen)]
test = adata[is_held_out]
test_inputs, _ = _split_controls(test, seen[is_held_out])
test_batches = np.unique(batch_labels(test_inputs))  # the order of _bridge_data's batches
print(f"held out batches {sorted(held_out)}: {test.n_obs:,} cells; training on {train.n_obs:,} cells")

## Held-out loss

In [ ]:
bridge = Standardised(NeuralBridgeTransfer(per_perturbation=True, **NEURAL))


@cache
def held_out_pairs():
    """Held-out labels, batches, raw targets, weights and standardised control inputs.

    Cached on first use, which is during fit: Standardised has its training means by then.
    """
    pairs = _bridge_data(bridge._standardise(test_inputs))
    labels = pairs.perturbations[pairs.pert]
    known = np.isin(labels, bridge.perturbations_)
    return labels[known], pairs.batch[known], pairs.y[known], pairs.weight[known], pairs.ctrl


def held_out_loss(predict) -> float:
    """Cell-weighted squared error over held-out pairs, with predictions in target units."""
    labels, batches, y, weight, _ = held_out_pairs()
    pred = np.empty_like(y)
    for b in np.unique(batches):
        pred[batches == b] = predict(b, labels[batches == b])
    return (weight * ((pred - y) ** 2).mean(axis=1)).sum() / weight.sum()


def bridge_predict(b, labels):
    net = bridge.model
    return net.bridge_.predict(net._onehot(_positions(net.perturbations_, labels)), held_out_pairs()[-1][b])


def linear_predict(model):
    def predict(b, labels):
        cells = test_inputs[batch_labels(test_inputs) == test_batches[b]]
        return model.predict(labels, cells)

    return predict

## Train

`_train` only reports the loss to W&B, so `_training_loss` is wrapped for this one fit. `len(val) == n_steps + 1`: `val[t]` and `train_loss[t]` use the $\psi$ from before update `t`, and the last `val` is the final model.

In [ ]:
train_loss, val = [], []
training_loss = NeuralPerXEmbeddingBridge._training_loss


def traced_loss(self, data):
    self._solve_A(data)
    val.append(held_out_loss(bridge_predict))
    loss = training_loss(self, data)
    train_loss.append(loss.item())
    return loss


with patch.object(NeuralPerXEmbeddingBridge, "_training_loss", traced_loss):
    bridge.fit(train)
val.append(held_out_loss(bridge_predict))

linear = _fit_linear(train)
references = {name: held_out_loss(linear_predict(m)) for name, m in linear.items()}

## Loss curves

Separate panels because the training and held-out losses are over different batches. Dashed lines are the linear models' held-out loss.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
left.plot(train_loss, color="#2a78d6", linewidth=2)
left.set(title="Training loss (cross-fitted)", xlabel="step", ylabel="weighted mse (unscaled targets)")

right.plot(val, color="#2a78d6", linewidth=2, label="per-perturbation bridge")
for (name, loss), dash in zip(references.items(), [(1, 2), (5, 2), (8, 2, 1, 2)], strict=True):
    right.axhline(loss, color="#52514e", linewidth=1.25, dashes=dash, label=name)
right.set(title="Held-out loss", xlabel="step", ylabel="weighted mse (unscaled targets)")
right.legend(frameon=False)

for ax in (left, right):
    ax.grid(axis="y", color="#e5e4e0", linewidth=0.75)
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
best = int(np.argmin(val))
pd.Series(
    {
        "training loss, first step": train_loss[0],
        "training loss, last step": train_loss[-1],
        "held-out loss, untrained psi": val[0],
        f"held-out loss, best (step {best})": val[best],
        "held-out loss, final": val[-1],
    }
    | {f"held-out loss, {name}": loss for name, loss in references.items()}
).to_frame("loss").style.format("{:.5f}")

**Reading it:** the bridge is training if the training loss falls clearly beyond its fold-to-fold noise. It is learning something useful only if the held-out loss also falls below its step-0 value. If the training loss falls while the held-out loss rises, that is overfitting across batches; if neither moves, the optimiser is not changing $\psi$ in a way that matters.

## Final scores

The metrics of the earlier runs, on effects (perturbation mean minus batch-matched control mean), for the final model.

In [ ]:
models = linear | {"per-perturbation bridge": bridge}
scores = score(models, test, TOP_K, seen[is_held_out])
scores.groupby(["metric", "model"], sort=False)["value"].mean().unstack("model").style.format("{:.4g}")